In [30]:
import pandas as pd
import re
import emoji
import nltk

nltk.download('punkt')

# === File paths ===
input_path = r"C:\Users\jange\Downloads\Sentiment_Data\Sentiment_Data.csv"
abbrev_path = r"C:\Users\jange\Downloads\Sentiment_Data\abbreviations_eng.csv"
output_path = r"C:\Users\jange\Downloads\Sentiment_Data\Sentiment_Data_Cleaned.csv"

# === Load data ===
df = pd.read_csv(input_path, encoding='latin1')
text_column = 'Tweet'

# === Load abbreviation dictionary with correct separator ===
abbrev_df = pd.read_csv(abbrev_path, sep=';', encoding='latin1')

# Create abbreviation dictionary
abbrev_dict = dict(zip(abbrev_df['abbr'], abbrev_df['long']))

def preprocess_text(text):
    if pd.isnull(text):
        return ""

    text = str(text).lower()

    # Remove IP addresses
    text = re.sub(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', '', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Remove emails
    text = re.sub(r'\S+@\S+', '', text)

    # Remove @mentions
    text = re.sub(r'@\w+', '', text)

    # Convert emojis to words
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = text.replace("_", " ")

    # Remove unwanted characters except spaces and alphanumeric
    text = re.sub(r'[^a-z0-9\s]', '', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Replace abbreviations after splitting
    words = text.split()
    replaced_words = [abbrev_dict.get(word, word) for word in words]
    cleaned_text = ' '.join(replaced_words)

    return cleaned_text

# Apply preprocessing
df['cleaned_text'] = df[text_column].apply(preprocess_text)

# Save cleaned data
df.to_csv(output_path, index=False)

print("✅ Preprocessing complete! Cleaned file saved at:")
print(output_path)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jange\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


✅ Preprocessing complete! Cleaned file saved at:
C:\Users\jange\Downloads\Sentiment_Data\Sentiment_Data_Cleaned.csv


In [31]:
import pandas as pd
from collections import Counter

# Load the cleaned data
output_path = r"C:\Users\jange\Downloads\Sentiment_Data\Sentiment_Data_Cleaned.csv"
df = pd.read_csv(output_path, encoding='latin1')

print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print()

# Show a few samples side by side: original and cleaned
print("Sample texts (Original vs Cleaned):")
print(df[['Tweet', 'cleaned_text']].sample(5, random_state=42).to_string(index=False))
print()

# Check for missing or empty cleaned texts
num_missing = df['cleaned_text'].isnull().sum()
num_empty = (df['cleaned_text'].str.strip() == '').sum()
print(f"Missing cleaned_text rows: {num_missing}")
print(f"Empty cleaned_text rows: {num_empty}")
print()

# Calculate word counts in cleaned text
df['word_count'] = df['cleaned_text'].apply(lambda x: len(str(x).split()))
print("Cleaned text word count statistics:")
print(df['word_count'].describe())
print()

# Top 10 most common words in cleaned text
all_words = ' '.join(df['cleaned_text'].dropna()).split()
word_freq = Counter(all_words)
print("Top 10 most frequent words:")
print(word_freq.most_common(10))


Total rows: 451332
Total columns: 3

Sample texts (Original vs Cleaned):
                                                                                                                                                                                              Tweet                                                                                                                                                                                 cleaned_text
                                                                                         stupid fucking freedom convoy is going past the store i work at stop fucking honking itÃ¢ÂÂs so annoying                                                                                         stupid fucking freedom convoy is going past the store i work at stop fucking honking its so annoying
                                                                                                Freedom Convoy coming to a country near you! Starting March 1

In [33]:
print(f"Before drop: {len(df)} rows")
df = df.dropna(subset=['cleaned_text'])
print(f"After drop: {len(df)} rows")


Before drop: 451332 rows
After drop: 450899 rows


In [34]:
import pandas as pd

# Load the cleaned file
file_path = r"C:\Users\jange\Downloads\Sentiment_Data\Sentiment_Data_Cleaned.csv"
df = pd.read_csv(file_path, encoding='latin1')

# Show count of each class in the 'Sentiment' column
class_counts = df['Sentiment'].value_counts()
print("Class distribution:\n", class_counts)


Class distribution:
 Sentiment
Strong_Pos    233700
Neutral        77016
Mild_Pos       64004
Strong_Neg     42556
Mild_Neg       34056
Name: count, dtype: int64


In [35]:
import pandas as pd

file_path = r"C:\Users\jange\Downloads\Sentiment_Data\Sentiment_Data_Cleaned.csv"
df = pd.read_csv(file_path, encoding='latin1')

# Define mapping
mapping = {
    'Strong_Pos': 'Positive',
    'Mild_Pos': 'Positive',
    'Neutral': 'Neutral',
    'Strong_Neg': 'Negative',
    'Mild_Neg': 'Negative'
}

# Create new simplified sentiment column
df['sentiment_simple'] = df['Sentiment'].map(mapping)

# Check distribution
print(df['sentiment_simple'].value_counts())

# Optionally save back
df.to_csv(file_path, index=False)


sentiment_simple
Positive    297704
Neutral      77016
Negative     76612
Name: count, dtype: int64


In [36]:
import pandas as pd

file_path = r"C:\Users\jange\Downloads\Sentiment_Data\Sentiment_Data_Cleaned.csv"
df = pd.read_csv(file_path, encoding='latin1')

# Check class counts
print("Original class distribution:")
print(df['sentiment_simple'].value_counts())

# Number of samples per class to keep
sample_size = 20000

# Sample from each class
balanced_df = pd.concat([
    df[df['sentiment_simple'] == label].sample(sample_size, random_state=42)
    for label in df['sentiment_simple'].unique()
])

# Shuffle the balanced dataset
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nBalanced class distribution:")
print(balanced_df['sentiment_simple'].value_counts())

# Save balanced dataset
balanced_file_path = r"C:\Users\jange\Downloads\Sentiment_Data\Sentiment_Data_Cleaned_Balanced.csv"
balanced_df.to_csv(balanced_file_path, index=False)

print(f"\nBalanced dataset saved at: {balanced_file_path}")


Original class distribution:
sentiment_simple
Positive    297704
Neutral      77016
Negative     76612
Name: count, dtype: int64

Balanced class distribution:
sentiment_simple
Positive    20000
Neutral     20000
Negative    20000
Name: count, dtype: int64

Balanced dataset saved at: C:\Users\jange\Downloads\Sentiment_Data\Sentiment_Data_Cleaned_Balanced.csv


In [37]:
import pandas as pd
from collections import Counter

balanced_file_path = r"C:\Users\jange\Downloads\Sentiment_Data\Sentiment_Data_Cleaned_Balanced.csv"
df = pd.read_csv(balanced_file_path, encoding='latin1')

print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}\n")

print("Sample texts (Original vs Cleaned):")
print(df[['Tweet', 'cleaned_text', 'sentiment_simple']].sample(5, random_state=42).to_string(index=False))
print()

num_missing = df['cleaned_text'].isnull().sum()
num_empty = (df['cleaned_text'].str.strip() == '').sum()
print(f"Missing cleaned_text rows: {num_missing}")
print(f"Empty cleaned_text rows: {num_empty}\n")

df['word_count'] = df['cleaned_text'].apply(lambda x: len(str(x).split()))
print("Cleaned text word count statistics:")
print(df['word_count'].describe())
print()

all_words = ' '.join(df['cleaned_text'].dropna()).split()
word_freq = Counter(all_words)
print("Top 10 most frequent words:")
print(word_freq.most_common(10))

print("\nClass distribution:")
print(df['sentiment_simple'].value_counts())


Total rows: 60000
Total columns: 4

Sample texts (Original vs Cleaned):
                                                                                                                                                                                                                                                                                       Tweet                                                                                                                                                                                                                cleaned_text sentiment_simple
                                                                                                                                                    @TBlashko The amount of hate and bigotry we've been seeing from supporters of the so-called "Freedom Convoy" is awful. I'm sorry, Tyler.                                                                                                      the amount of hate a

In [39]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load balanced data
balanced_file_path = r"C:\Users\jange\Downloads\Sentiment_Data\Sentiment_Data_Cleaned_Balanced.csv"
df = pd.read_csv(balanced_file_path, encoding='latin1')

# First split off test set (15%)
train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['sentiment_simple'], random_state=42)

# Then split train and validation (from the remaining 85%, val is 15% total → ~0.176 of train_val)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.176, stratify=train_val_df['sentiment_simple'], random_state=42)

# Save to csv files
train_df.to_csv(r"C:\Users\jange\Downloads\Sentiment_Data\train.csv", index=False)
val_df.to_csv(r"C:\Users\jange\Downloads\Sentiment_Data\validation.csv", index=False)
test_df.to_csv(r"C:\Users\jange\Downloads\Sentiment_Data\test.csv", index=False)

print("✅ Data split complete and files saved:")
print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size: {len(test_df)}")


✅ Data split complete and files saved:
Train size: 42024
Validation size: 8976
Test size: 9000
